# Setup

In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import geopandas as gpd
from dask.distributed import Client, LocalCluster
import dask.dataframe as dd
import hvplot.dask
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

In [2]:
dir = 'G:/My Drive/Work/ADB/SAR/'

## Dask Cluster

In [3]:
cluster = LocalCluster(n_workers=5, 
                       threads_per_worker=1,
                       memory_limit='4GB')
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 5
Total threads: 5,Total memory: 18.63 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:55736,Workers: 5
Dashboard: http://127.0.0.1:8787/status,Total threads: 5
Started: Just now,Total memory: 18.63 GiB
Comm: tcp://127.0.0.1:55766,Total threads: 1
Dashboard: http://127.0.0.1:55768/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:55739,


# Preprocessing

In [25]:
for region in ['NORTH', 'SOUTH']:
    print(region)
    df_pm = dd.read_csv(f'datasets/VN_{region}_VER2.csv')
    df_pm = df_pm[['Timestamp', 'lon', 'lat', 'PM25_mean', 'PM25_max']]

    df_pm_alt = dd.read_csv(os.path.join(dir, f'combined_datasets/GHAP_{region}.csv'))

    df = df_pm.merge(df_pm_alt, on=['Timestamp', 'lon', 'lat'], how='left')

    df.to_csv(os.path.join(dir, f'combined_datasets/PM25_{region}.csv'), single_file=True)

NORTH
SOUTH


PermissionError: [Errno 13] Permission denied: 'G:/My Drive/Work/ADB/SAR/combined_datasets/PM25_SOUTH.csv'

In [27]:
df = dd.read_csv(os.path.join(dir, f'combined_datasets/PM25_SOUTH.csv'))
df['year'] = df['Timestamp'].str[:4]
df = df[['year', 'PM25_mean', 'PM25_max', 'GHAP_PM25_mean', 'GHAP_PM25_max']]
df.to_csv(os.path.join(dir, f'combined_datasets/PM25_SOUTH.csv'), single_file=True)

['G:\\My Drive\\Work\\ADB\\SAR\\combined_datasets\\PM25_SOUTH.csv']

In [28]:
df = dd.read_csv(os.path.join(dir, f'combined_datasets/PM25_NORTH.csv'))
df['year'] = df['Timestamp'].str[:4]
df = df[['year', 'PM25_mean', 'PM25_max', 'GHAP_PM25_mean', 'GHAP_PM25_max']]
df.to_csv(os.path.join(dir, f'combined_datasets/PM25_NORTH.csv'), single_file=True)

['G:\\My Drive\\Work\\ADB\\SAR\\combined_datasets\\PM25_NORTH.csv']

# Analysis

In [5]:
df = dd.read_csv(os.path.join(dir, f'combined_datasets/PM25_SOUTH.csv'))
# df['year'] = df['Timestamp'].str[:4]

In [6]:
df.head(2)

,Unnamed: 0,year,PM25_mean,PM25_max,GHAP_PM25_mean,GHAP_PM25_max
0,0,2017,10.46,34.8,1.594,2.76
1,1,2017,10.46,34.8,1.618,3.52


In [21]:
df_y = df[df['year'] == 2018]

In [22]:
df_y.head(2)

,Unnamed: 0,year,PM25_mean,PM25_max,GHAP_PM25_mean,GHAP_PM25_max
1812,1812,2018,16.95,52.28,1.811,3.05
1813,1813,2018,16.95,52.28,1.912,3.28


In [23]:
df_y.hvplot.kde(y=['PM25_mean', 'GHAP_PM25_mean'], alpha=0.5, value_label='Mean Concentration', legend='top_right')

:NdOverlay   [Variable]
   :Distribution   [Mean Concentration]   (Density)

In [24]:
df_y.hvplot.kde(y=['PM25_max', 'GHAP_PM25_max'], alpha=0.5, value_label='Max Concentration', legend='top_right')

:NdOverlay   [Variable]
   :Distribution   [Max Concentration]   (Density)

In [25]:
for col in ['GHAP_PM25_mean', 'GHAP_PM25_max']:
    df_y[col] = df_y[col]*10

In [26]:
df_y.hvplot.kde(y=['PM25_mean', 'GHAP_PM25_mean'], alpha=0.5, value_label='Mean Concentration', legend='top_right')

:NdOverlay   [Variable]
   :Distribution   [Mean Concentration]   (Density)

In [27]:
df_y.hvplot.kde(y=['PM25_max', 'GHAP_PM25_max'], alpha=0.5, value_label='Max Concentration', legend='top_right')

:NdOverlay   [Variable]
   :Distribution   [Max Concentration]   (Density)

In [85]:
df = pd.read_csv(os.path.join(dir, f'combined_datasets/PM25_SOUTH.csv'))
df = df.drop(['Unnamed: 0'], axis=1)
# df['ind'] = df.index
df.head()

,year,PM25_mean,PM25_max,GHAP_PM25_mean,GHAP_PM25_max
0,2017,10.460,34.80,1.594,2.76
1,2017,10.460,34.80,1.618,3.52
2,2017,10.460,34.80,1.500,2.52
3,2017,9.695,33.06,1.659,2.79
4,2017,9.695,33.06,1.639,2.62


In [86]:
df = df[~df['GHAP_PM25_mean'].isnull()]
df

,year,PM25_mean,PM25_max,GHAP_PM25_mean,GHAP_PM25_max
0,2017,10.460,34.80,1.5940,2.76
1,2017,10.460,34.80,1.6180,3.52
2,2017,10.460,34.80,1.5000,2.52
3,2017,9.695,33.06,1.6590,2.79
4,2017,9.695,33.06,1.6390,2.62
...,...,...,...,...,...
8194736,2020,6.430,33.34,0.9660,1.30
8194737,2020,6.430,33.34,0.9890,1.34
8194738,2020,10.070,41.50,1.0590,1.44
8194739,2020,6.430,33.34,0.8135,1.10


In [89]:
df_samp = df.sample(frac=0.001, random_state=60)
# df_samp['year'] = df_samp['year'].astype(str)

year = pd.Categorical(df_samp['year'])
df_samp.set_index(['year'])
df_samp['year'] = year

Y = df_samp['GHAP_PM25_mean']
X = df_samp['PM25_mean']
X = sm.add_constant(X)
model = sm.OLS(Y, X)
results = model.fit(cov_type='HC1')
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:         GHAP_PM25_mean   R-squared:                       0.289
Model:                            OLS   Adj. R-squared:                  0.289
Method:                 Least Squares   F-statistic:                 2.473e+04
Date:                Tue, 02 Jul 2024   Prob (F-statistic):               0.00
Time:                        23:47:54   Log-Likelihood:                -62556.
No. Observations:               81940   AIC:                         1.251e+05
Df Residuals:                   81938   BIC:                         1.251e+05
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.1787      0.003    343.458      0.0

In [90]:
Y = df_samp['GHAP_PM25_max']
X = df_samp['PM25_max']
X = sm.add_constant(X)
model = sm.OLS(Y, X)
results = model.fit(cov_type='HC1')
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:          GHAP_PM25_max   R-squared:                       0.189
Model:                            OLS   Adj. R-squared:                  0.189
Method:                 Least Squares   F-statistic:                     5434.
Date:                Tue, 02 Jul 2024   Prob (F-statistic):               0.00
Time:                        23:47:55   Log-Likelihood:            -1.1697e+05
No. Observations:               81940   AIC:                         2.339e+05
Df Residuals:                   81938   BIC:                         2.340e+05
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.8438      0.011    169.256      0.0

In [91]:
df_samp['log_pm25_mean'] = np.log(df_samp['PM25_mean'])
df_samp['log_pm25_max'] = np.log(df_samp['PM25_max'])

Y = df_samp['GHAP_PM25_mean']
X = df_samp['log_pm25_mean']
X = sm.add_constant(X)
model = sm.OLS(Y, X)
results = model.fit(cov_type='HC1')
print(results.summary())


Y = df_samp['GHAP_PM25_max']
X = df_samp['log_pm25_max']
X = sm.add_constant(X)
model = sm.OLS(Y, X)
results = model.fit(cov_type='HC1')
print(results.summary())



                            OLS Regression Results                            
Dep. Variable:         GHAP_PM25_mean   R-squared:                       0.352
Model:                            OLS   Adj. R-squared:                  0.352
Method:                 Least Squares   F-statistic:                 5.773e+04
Date:                Tue, 02 Jul 2024   Prob (F-statistic):               0.00
Time:                        23:50:06   Log-Likelihood:                -58730.
No. Observations:               81940   AIC:                         1.175e+05
Df Residuals:                   81938   BIC:                         1.175e+05
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.0719      0.007    -10.355

# Close Dask Cluster

In [29]:
client.close()
cluster.close()